# 🍎 Fresh Fruit Detection — MobileNetV2 Transfer Learning
**Student:** Anushka Marcelo | **MCA Part 1, Sem 2** | **Goa Business School, Goa University**

---
### ⚡ INSTRUCTIONS — Run each cell one by one from top to bottom
- Click ▶ on each cell
- Wait for it to finish (✅ appears) before running the next
- If any error occurs, re-run that same cell once more

## 📦 CELL 1 — Install Libraries

In [ ]:
!pip install kaggle -q

import os, numpy as np, matplotlib.pyplot as plt
import matplotlib.image as mpimg, seaborn as sns, warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix

print('✅ TensorFlow:', tf.__version__)
print('✅ GPU:', tf.config.list_physical_devices('GPU'))
print('✅ All libraries ready!')

## 📂 CELL 2 — Upload kaggle.json & Download Dataset

In [ ]:
from google.colab import files

print('📤 Upload your kaggle.json file when prompted...')
uploaded = files.upload()

# Setup Kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print('✅ Kaggle credentials set!')

## 📥 CELL 3 — Download & Extract Dataset

In [ ]:
# Download dataset
!kaggle datasets download -d sriramr/fruits-fresh-and-rotten-for-classification

# Unzip
!unzip -q fruits-fresh-and-rotten-for-classification.zip -d /content/fruit_dataset

print('✅ Dataset downloaded and extracted!')
print('\n📁 Folder structure:')
for root, dirs, files_list in os.walk('/content/fruit_dataset'):
    level = root.replace('/content/fruit_dataset', '').count(os.sep)
    if level <= 3:
        indent = '  ' * level
        n_imgs = len([f for f in files_list if f.endswith(('.jpg','.jpeg','.png'))])
        name = os.path.basename(root)
        if n_imgs > 0:
            print(f'{indent}📁 {name}/  → {n_imgs} images')
        elif level <= 2:
            print(f'{indent}📁 {name}/')

## 🔍 CELL 4 — Set Paths (Auto-detect)

In [ ]:
# Auto-detect the correct train/test paths
TRAIN_DIR = None
TEST_DIR  = None

for root, dirs, files_list in os.walk('/content/fruit_dataset'):
    for d in dirs:
        full_path = os.path.join(root, d)
        if d.lower() == 'train' and TRAIN_DIR is None:
            TRAIN_DIR = full_path
        if d.lower() == 'test' and TEST_DIR is None:
            TEST_DIR = full_path

print(f'✅ TRAIN_DIR: {TRAIN_DIR}')
print(f'✅ TEST_DIR:  {TEST_DIR}')
print(f'\nClasses in training set:')
for c in sorted(os.listdir(TRAIN_DIR)):
    n = len(os.listdir(os.path.join(TRAIN_DIR, c)))
    print(f'  {c}: {n} images')

## 📊 CELL 5 — Dataset Analysis & Visualization

In [ ]:
# Count images per class
class_counts_train = {}
class_counts_test  = {}

for cls in os.listdir(TRAIN_DIR):
    p = os.path.join(TRAIN_DIR, cls)
    if os.path.isdir(p):
        class_counts_train[cls] = len(os.listdir(p))

for cls in os.listdir(TEST_DIR):
    p = os.path.join(TEST_DIR, cls)
    if os.path.isdir(p):
        class_counts_test[cls] = len(os.listdir(p))

total_train = sum(class_counts_train.values())
total_test  = sum(class_counts_test.values())

print('='*50)
print('📊 DATASET SUMMARY')
print('='*50)
print(f'Total Training Images : {total_train}')
print(f'Total Testing Images  : {total_test}')
print(f'Total Images          : {total_train + total_test}')
print(f'Number of Classes     : {len(class_counts_train)}')

## 📈 CELL 6 — Bar Chart: Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Dataset Class Distribution', fontsize=16, fontweight='bold')

classes = list(class_counts_train.keys())
counts  = list(class_counts_train.values())
colors  = ['#2ecc71' if 'fresh' in c.lower() else '#e74c3c' for c in classes]

bars = axes[0].bar(classes, counts, color=colors, edgecolor='white')
axes[0].set_title('Training Set', fontweight='bold')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Number of Images')
axes[0].tick_params(axis='x', rotation=45)
for bar, count in zip(bars, counts):
    axes[0].text(bar.get_x()+bar.get_width()/2., bar.get_height()+5,
                 str(count), ha='center', va='bottom', fontweight='bold', fontsize=9)

t_classes = list(class_counts_test.keys())
t_counts  = list(class_counts_test.values())
t_colors  = ['#27ae60' if 'fresh' in c.lower() else '#c0392b' for c in t_classes]
bars2 = axes[1].bar(t_classes, t_counts, color=t_colors, edgecolor='white')
axes[1].set_title('Test Set', fontweight='bold')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Number of Images')
axes[1].tick_params(axis='x', rotation=45)
for bar, count in zip(bars2, t_counts):
    axes[1].text(bar.get_x()+bar.get_width()/2., bar.get_height()+2,
                 str(count), ha='center', va='bottom', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved!')

## 🥧 CELL 7 — Pie Chart: Fresh vs Rotten

In [ ]:
fresh_count  = sum(v for k,v in class_counts_train.items() if 'fresh' in k.lower())
rotten_count = sum(v for k,v in class_counts_train.items() if 'rotten' in k.lower())

fig, ax = plt.subplots(figsize=(7,7))
wedges, texts, autotexts = ax.pie(
    [fresh_count, rotten_count],
    labels=['Fresh Fruits', 'Rotten Fruits'],
    autopct='%1.1f%%',
    colors=['#2ecc71','#e74c3c'],
    explode=(0.05, 0.05),
    startangle=140,
    textprops={'fontsize':13}
)
for a in autotexts: a.set_fontweight('bold')
ax.set_title('Fresh vs Rotten (Training Set)', fontsize=15, fontweight='bold')
plt.savefig('fresh_vs_rotten_pie.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Pie chart saved!')

## 🖼️ CELL 8 — Sample Images from Each Class

In [ ]:
import random
class_dirs = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
n_classes  = len(class_dirs)

fig, axes = plt.subplots(n_classes, 4, figsize=(14, n_classes * 3))
fig.suptitle('Sample Images from Each Class', fontsize=16, fontweight='bold')

for row, cls in enumerate(class_dirs):
    cls_path = os.path.join(TRAIN_DIR, cls)
    images   = [f for f in os.listdir(cls_path) if f.endswith(('.jpg','.jpeg','.png'))]
    sample   = random.sample(images, min(4, len(images)))
    for col, img_name in enumerate(sample):
        img = mpimg.imread(os.path.join(cls_path, img_name))
        ax  = axes[row][col] if n_classes > 1 else axes[col]
        ax.imshow(img)
        ax.axis('off')
        if col == 0:
            color = 'green' if 'fresh' in cls.lower() else 'red'
            ax.set_title(cls, fontsize=10, fontweight='bold', color=color)

plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Sample images saved!')

## ⚙️ CELL 9 — Preprocessing & Data Augmentation

In [ ]:
IMG_SIZE   = 224
BATCH_SIZE = 32
EPOCHS     = 15
LR         = 0.0001

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    subset='training', shuffle=True
)
val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    subset='validation', shuffle=False
)
test_generator = test_datagen.flow_from_directory(
    TEST_DIR, target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    shuffle=False
)

NUM_CLASSES  = len(train_generator.class_indices)
class_labels = list(train_generator.class_indices.keys())
print('✅ Data generators ready!')
print(f'Classes ({NUM_CLASSES}):', class_labels)
print(f'Training batches  : {len(train_generator)}')
print(f'Validation batches: {len(val_generator)}')
print(f'Test batches      : {len(test_generator)}')

## 🧠 CELL 10 — Build MobileNetV2 Model

In [ ]:
# Load pretrained MobileNetV2 (no top layer)
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Freeze base layers

# Add custom classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)
model.compile(
    optimizer=Adam(learning_rate=LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('✅ Model built!')
print(f'Total params     : {model.count_params():,}')
print(f'Base model layers: {len(base_model.layers)}')

## 🏋️ CELL 11 — Train the Model

In [ ]:
early_stop  = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
reduce_lr   = ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-7, verbose=1)
checkpoint  = ModelCheckpoint('best_fruit_model.keras', monitor='val_accuracy', save_best_only=True, verbose=1)

print('🚀 Training started...\n')
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=[early_stop, reduce_lr, checkpoint],
    verbose=1
)
print('\n✅ Training complete!')

## 📈 CELL 12 — Accuracy & Loss Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model Training Performance', fontsize=16, fontweight='bold')
ep = range(1, len(history.history['accuracy'])+1)

axes[0].plot(ep, history.history['accuracy'],     'b-o', label='Train', linewidth=2)
axes[0].plot(ep, history.history['val_accuracy'], 'r-o', label='Validation', linewidth=2)
axes[0].set_title('Accuracy', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3); axes[0].set_ylim([0,1])

axes[1].plot(ep, history.history['loss'],     'b-o', label='Train', linewidth=2)
axes[1].plot(ep, history.history['val_loss'], 'r-o', label='Validation', linewidth=2)
axes[1].set_title('Loss', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Training curves saved!')

## 📊 CELL 13 — Evaluate Model (Confusion Matrix + Report)

In [ ]:
from tensorflow.keras.models import load_model
best_model = load_model('best_fruit_model.keras')

test_loss, test_acc = best_model.evaluate(test_generator, verbose=1)
print(f'\n✅ Test Accuracy : {test_acc*100:.2f}%')
print(f'✅ Test Loss     : {test_loss:.4f}')

test_generator.reset()
y_pred = np.argmax(best_model.predict(test_generator, verbose=1), axis=1)
y_true = test_generator.classes

print('\n' + '='*55)
print('📋 CLASSIFICATION REPORT')
print('='*55)
print(classification_report(y_true, y_pred, target_names=class_labels))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=class_labels, yticklabels=class_labels,
            cmap='Blues', linewidths=0.5)
plt.title('Confusion Matrix', fontsize=15, fontweight='bold')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Confusion matrix saved!')

## CELL 14 — DEMO: Predict on Test Images

In [ ]:
import random

def predict_fruit(image_path, model, class_labels, img_size=224):
    img       = load_img(image_path, target_size=(img_size, img_size))
    arr       = np.expand_dims(img_to_array(img) / 255.0, axis=0)
    preds     = model.predict(arr, verbose=0)
    idx       = np.argmax(preds[0])
    return class_labels[idx], preds[0][idx]*100, preds[0]

all_test_classes = [d for d in os.listdir(TEST_DIR) if os.path.isdir(os.path.join(TEST_DIR, d))]
n_show = min(len(all_test_classes), 6)
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
fig.suptitle('🍎 Demo Predictions', fontsize=16, fontweight='bold')

for i, cls in enumerate(random.sample(all_test_classes, n_show)):
    cls_path = os.path.join(TEST_DIR, cls)
    imgs     = [f for f in os.listdir(cls_path) if f.endswith(('.jpg','.jpeg','.png'))]
    img_path = os.path.join(cls_path, random.choice(imgs))
    pred_label, conf, _ = predict_fruit(img_path, best_model, class_labels)
    correct  = cls == pred_label
    img      = load_img(img_path, target_size=(224,224))
    ax       = axes[i//3][i%3]
    ax.imshow(img); ax.axis('off')
    color  = '#2ecc71' if correct else '#e74c3c'
    status = '✅' if correct else '❌'
    ax.set_title(f'{status} Pred: {pred_label}\nConf: {conf:.1f}% | True: {cls}',
                 fontsize=9, color=color, fontweight='bold')

plt.tight_layout()
plt.savefig('demo_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Demo predictions saved!')

## 📤 CELL 15 — Upload Your Own Fruit Image!

In [ ]:
from google.colab import files
print('📤 Upload any fruit image!')
uploaded_demo = files.upload()

for filename in uploaded_demo.keys():
    pred_label, conf, all_probs = predict_fruit(filename, best_model, class_labels)
    img = load_img(filename, target_size=(224,224))
    plt.figure(figsize=(6,6))
    plt.imshow(img); plt.axis('off')
    is_fresh = 'fresh' in pred_label.lower()
    color    = '#2ecc71' if is_fresh else '#e74c3c'
    emoji    = '✅ FRESH' if is_fresh else '❌ ROTTEN'
    plt.title(f'{emoji}\n{pred_label} ({conf:.1f}% confidence)',
              fontsize=14, fontweight='bold', color=color)
    plt.tight_layout(); plt.show()

    print('\n📋 All class probabilities:')
    for label, prob in sorted(zip(class_labels, all_probs), key=lambda x: -x[1]):
        bar = '█' * int(prob*30)
        print(f'  {label:<22} {bar} {prob*100:.2f}%')

## 💾 CELL 16 — Save & Download Everything

In [ ]:
best_model.save('fresh_fruit_detector_final.keras')
print('✅ Model saved!')

from google.colab import files
for f in ['fresh_fruit_detector_final.keras', 'class_distribution.png',
          'fresh_vs_rotten_pie.png', 'sample_images.png',
          'training_curves.png', 'confusion_matrix.png', 'demo_predictions.png']:
    if os.path.exists(f):
        files.download(f)
        print(f'  ⬇️  {f}')

print('\n✅ All files downloaded!')

## 📝 CELL 17 — Final Summary

In [ ]:
print('='*60)
print('     🍎 FRESH FRUIT DETECTION — PROJECT SUMMARY')
print('='*60)
print(f'Student    : Anushka Marcelo')
print(f'Course     : MCA Part 1, Semester 2')
print(f'College    : Goa Business School, Goa University')
print('─'*60)
print(f'Model      : MobileNetV2 (Transfer Learning)')
print(f'Dataset    : Fresh and Rotten Fruits (Kaggle)')
print(f'Classes    : {NUM_CLASSES} ({class_labels})')
print(f'Train Imgs : {total_train}')
print(f'Test Imgs  : {total_test}')
print('─'*60)
print(f'Test Accuracy : {test_acc*100:.2f}%')
print(f'Test Loss     : {test_loss:.4f}')
print('─'*60)
print('Techniques Used:')
print('  ✅ Transfer Learning (MobileNetV2 + ImageNet weights)')
print('  ✅ Data Augmentation (rotation, flip, zoom, shift)')
print('  ✅ Batch Normalization & Dropout')
print('  ✅ EarlyStopping & ReduceLROnPlateau')
print('  ✅ Confusion Matrix & Classification Report')
print('─'*60)
print('Future Improvements:')
print('  🔹 Mobile app deployment (TFLite)')
print('  🔹 Real-time webcam detection')
print('  🔹 Ripeness stage detection')
print('  🔹 REST API for retail systems')
print('='*60)